<a href="https://colab.research.google.com/github/Venu-max/NASSCOM-AI/blob/main/Day10_U21_%C2%B7_ETL_%26_Orchestration_%E2%80%94_Hands_on.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Idea of this notebook: good data engineering is reliability engineering — re-runnable pipelines, validated data, and orchestration you can trust. We build the plumbing on tiny synthetic data using only Python, pandas and SQLite (no servers needed).

Small practicals:

The ETL pipeline — extract → transform → load (messy CSV → clean SQLite table)
Idempotency — naive append vs upsert (re-run safely, no duplicates)
Incremental extract — pull only what changed (high-water mark)
Data-quality validation — fail fast, quarantine bad rows
Partitioning & backfills — overwrite one date, leave the rest untouched
Orchestration — a mini DAG with topological order + retries
Every line of code is commented. Run the cells top to bottom.

In [1]:
import pandas as pd                      # pandas: tabular data handling
import sqlite3                             # sqlite3: a built-in SQL database (no install)
from io import StringIO                    # lets us read a CSV from a string in memory
from datetime import datetime              # for parsing/formatting dates
import time                                # for retry backoff timing

In [ ]:
#1. The ETL pipeline (Extract → Transform → Load)

In [2]:
# --- EXTRACT: pretend this CSV arrived from a source system (raw + messy) ---
raw_csv = """order_id,order_date,amount,region
1001,12/05/2024, $1200 ,north
1002,2024-05-13,980.50,North
1003,13/05/2024,N/A,SOUTH
1001,12/05/2024, $1200 ,north
1004,14/05/2024, 2300 , South
"""
df_raw = pd.read_csv(StringIO(raw_csv), skipinitialspace=True)  # read CSV; trim spaces after commas
print("RAW (as extracted):")
print(df_raw)

# --- TRANSFORM: parse, cast, default, standardise, dedupe ---
def parse_date(s):                                  # turn mixed date formats into ISO YYYY-MM-DD
    s = str(s).strip()                              # remove surrounding whitespace
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):            # try day-first, then ISO
        try:
            return datetime.strptime(s, fmt).strftime("%Y-%m-%d")  # reformat to ISO
        except ValueError:
            continue                                # not this format; try the next
    return None                                     # unknown format -> null

def clean_amount(s):                                # turn " $1200 " / "N/A" into a float
    if pd.isna(s):                                  # pandas may already parse "N/A" as NaN on read
        return 0.0                                  # default missing amounts to 0.0
    s = str(s).strip().replace("$", "").replace(",", "").strip()   # drop $, commas, spaces
    if s.upper() in ("", "N/A", "NA", "NULL", "NONE"):             # missing markers
        return 0.0                                  # default missing amounts to 0.0
    return float(s)                                 # cast the cleaned string to a number

df = df_raw.copy()                                  # work on a copy
df["order_date"] = df["order_date"].apply(parse_date)   # standardise dates
df["amount"]     = df["amount"].apply(clean_amount)     # clean + cast amounts
df["region"]     = df["region"].str.strip().str.title()  # standardise region capitalisation
df = df.drop_duplicates(subset="order_id", keep="first").reset_index(drop=True)  # dedupe by id
print("\nCLEAN (after transform):")
print(df)

# --- LOAD: write the clean table into SQLite, then read it back to confirm ---
conn = sqlite3.connect(":memory:")                  # in-memory database for the demo
df.to_sql("orders", conn, index=False, if_exists="replace")  # load the clean DataFrame
print("\nLOADED into SQLite — row count:",
      conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0])

RAW (as extracted):
   order_id  order_date  amount region
0      1001  12/05/2024  $1200   north
1      1002  2024-05-13  980.50  North
2      1003  13/05/2024     NaN  SOUTH
3      1001  12/05/2024  $1200   north
4      1004  14/05/2024   2300   South

CLEAN (after transform):
   order_id  order_date  amount region
0      1001  2024-05-12  1200.0  North
1      1002  2024-05-13   980.5  North
2      1003  2024-05-13     0.0  South
3      1004  2024-05-14  2300.0  South

LOADED into SQLite — row count: 4


In [ ]:
#2. Idempotency (re-run safely, no duplicates)
#The golden rule: a pipeline you can re-run safely is one you can fix at 3 a.m. We compare a naive append (re-running creates duplicates) with an upsert by primary key (re-running is harmless).

In [3]:
cur = conn.cursor()                                 # a cursor on our in-memory DB

# --- Approach A: a plain table with naive INSERT (NOT idempotent) ---
cur.execute("CREATE TABLE sales_naive (order_id INT, amount REAL)")   # no primary key
def load_naive(rows):
    cur.executemany("INSERT INTO sales_naive VALUES (?, ?)", rows)    # plain insert = append
    conn.commit()

# --- Approach B: a PRIMARY KEY + INSERT OR REPLACE (idempotent upsert) ---
cur.execute("CREATE TABLE sales_upsert (order_id INT PRIMARY KEY, amount REAL)")  # PK on id
def load_upsert(rows):
    cur.executemany("INSERT OR REPLACE INTO sales_upsert VALUES (?, ?)", rows)  # upsert by id
    conn.commit()

batch = [(1, 100.0), (2, 200.0), (3, 300.0)]        # the same batch we will load twice

load_naive(batch);  load_naive(batch)               # run the naive load TWICE (e.g. a retry)
load_upsert(batch); load_upsert(batch)              # run the upsert load TWICE

n_naive  = cur.execute("SELECT COUNT(*) FROM sales_naive").fetchone()[0]   # count rows
n_upsert = cur.execute("SELECT COUNT(*) FROM sales_upsert").fetchone()[0]
print(f"Naive append, ran twice -> {n_naive} rows  (duplicates!)")
print(f"Upsert by PK, ran twice -> {n_upsert} rows  (idempotent — safe to retry)")

Naive append, ran twice -> 6 rows  (duplicates!)
Upsert by PK, ran twice -> 3 rows  (idempotent — safe to retry)


In [ ]:
#3. Incremental extract (high-water mark)
#Pulling the whole source every run is slow. Instead we remember the newest timestamp we've seen (the high-water mark) and next time pull only rows newer than it.

In [4]:
# A source table that grows over time; each row carries an 'updated_at' timestamp
source = pd.DataFrame({
    "id":         [1, 2, 3],                                  # business keys
    "value":      ["a", "b", "c"],                            # some payload
    "updated_at": pd.to_datetime(["2024-05-10", "2024-05-10", "2024-05-11"])  # change time
})

high_water = pd.Timestamp("1900-01-01")                      # start: we've seen nothing yet

# --- RUN 1: full pull (everything is newer than 1900) ---
new = source[source["updated_at"] > high_water]              # rows newer than the mark
print("Run 1 pulled:", new["id"].tolist())
high_water = new["updated_at"].max()                         # advance the high-water mark

# New data lands in the source between runs
source = pd.concat([source, pd.DataFrame({
    "id": [4, 5], "value": ["d", "e"],
    "updated_at": pd.to_datetime(["2024-05-12", "2024-05-12"])})], ignore_index=True)

# --- RUN 2: incremental pull (only rows newer than the mark) ---
new = source[source["updated_at"] > high_water]              # just the fresh rows
print("Run 2 pulled:", new["id"].tolist(), " (only the new rows — cheap!)")
high_water = max(high_water, new["updated_at"].max())        # advance the mark again

Run 1 pulled: [1, 2, 3]
Run 2 pulled: [4, 5]  (only the new rows — cheap!)


In [ ]:
#4. Data-quality validation (fail fast, quarantine)
#A bad row caught at ingestion costs minutes; the same row in a board report costs trust. We run checks at the boundary, keep the good rows, and quarantine the bad ones with a reason.

In [5]:
# A batch with deliberate problems: a negative amount, a missing region, a dupe, a null amount
batch = pd.DataFrame({
    "order_id": [1, 2, 3, 3, 5],
    "amount":   [100.0, -50.0, 300.0, 300.0, None],
    "region":   ["North", "South", "East", "West", None],
})

def validate(df):                                   # attach a failure reason to each row
    reasons = [[] for _ in range(len(df))]          # one (initially empty) reason list per row
    seen = set()                                    # track ids we've already seen (uniqueness)
    for i, (_, row) in enumerate(df.iterrows()):    # walk the rows in order
        if pd.isna(row["amount"]):                  # completeness: amount present?
            reasons[i].append("amount missing")
        elif row["amount"] < 0:                     # range/domain: amount non-negative?
            reasons[i].append("amount negative")
        if pd.isna(row["region"]):                  # completeness: region present?
            reasons[i].append("region missing")
        if row["order_id"] in seen:                 # uniqueness: primary key not repeated?
            reasons[i].append("duplicate id")
        seen.add(row["order_id"])                   # remember this id
    out = df.copy()
    out["_errors"] = ["; ".join(r) for r in reasons]  # join reasons into one string ("" = clean)
    return out

checked    = validate(batch)                        # run the checks
valid      = checked[checked["_errors"] == ""]      # rows that passed every check
quarantine = checked[checked["_errors"] != ""]      # rows that failed at least one check
print(f"VALID rows: {len(valid)}  |  QUARANTINED rows: {len(quarantine)}\n")
print("Quarantined (with reasons):")
print(quarantine[["order_id", "amount", "region", "_errors"]].to_string(index=False))

VALID rows: 2  |  QUARANTINED rows: 3

Quarantined (with reasons):
 order_id  amount region                        _errors
        2   -50.0  South                amount negative
        3   300.0   West                   duplicate id
        5     NaN   None amount missing; region missing


5. Partitioning & backfills
Storing data by date (one slice per dt) makes re-runs surgical: re-running a day overwrites only that partition, and a backfill just fills a missing historical day.

In [ ]:
#5. Partitioning & backfills

In [6]:
store = {}                                          # our partitioned store: dt -> DataFrame

def write_partition(dt, df):                        # OVERWRITE just this date's slice
    store[dt] = df.copy()                           # idempotent per partition (replace, not append)

def make_day(dt, n):                                # fabricate n rows for a given date
    return pd.DataFrame({"dt": [dt] * n, "amount": range(n)})

# Initial load of three days
for dt, n in [("2024-05-11", 3), ("2024-05-12", 2), ("2024-05-13", 4)]:
    write_partition(dt, make_day(dt, n))
print("After initial load :", {k: len(v) for k, v in sorted(store.items())})

# Re-run ONE day with corrected data -> only that partition changes
write_partition("2024-05-12", make_day("2024-05-12", 5))   # 05-12 corrected: now 5 rows
print("After re-running 05-12:", {k: len(v) for k, v in sorted(store.items())})

# BACKFILL a missing historical day
write_partition("2024-05-10", make_day("2024-05-10", 2))   # add the older day
print("After backfilling 05-10:", {k: len(v) for k, v in sorted(store.items())})
print("\nOther partitions were never touched — that is the point of partitioning.")

After initial load : {'2024-05-11': 3, '2024-05-12': 2, '2024-05-13': 4}
After re-running 05-12: {'2024-05-11': 3, '2024-05-12': 5, '2024-05-13': 4}
After backfilling 05-10: {'2024-05-10': 2, '2024-05-11': 3, '2024-05-12': 5, '2024-05-13': 4}

Other partitions were never touched — that is the point of partitioning.


In [ ]:
#6. Orchestration — a mini DAG with order + retries
#An orchestrator (like Airflow) models tasks and dependencies as a DAG, finds a valid run order, and retries transient failures. We build a tiny one. The DAG mirrors the slide: extract → (validate, dedupe) → transform → load → publish.

In [7]:
# Define the pipeline as a DAG: task -> list of upstream dependencies
deps = {
    "extract":   [],                                # no dependencies — can start first
    "validate":  ["extract"],                       # needs extract
    "dedupe":    ["extract"],                        # needs extract (runs alongside validate)
    "transform": ["validate", "dedupe"],            # needs both
    "load":      ["transform"],                      # needs transform
    "publish":   ["load"],                           # needs load
}

def topo_sort(deps):                                # Kahn's algorithm: a valid order, proves acyclic
    indeg = {t: len(d) for t, d in deps.items()}    # count unmet dependencies per task
    ready = [t for t, n in indeg.items() if n == 0] # tasks with no deps are ready to run
    order = []
    while ready:                                    # repeatedly take a ready task
        t = ready.pop(0)
        order.append(t)
        for u, d in deps.items():                   # any task depending on t loses one dep
            if t in d:
                indeg[u] -= 1
                if indeg[u] == 0:                   # all deps met -> now ready
                    ready.append(u)
    if len(order) != len(deps):                     # leftover tasks => a cycle exists
        raise ValueError("cycle detected — not a DAG!")
    return order

run_order = topo_sort(deps)                         # compute a safe execution order
print("Valid run order:", run_order)

Valid run order: ['extract', 'validate', 'dedupe', 'transform', 'load', 'publish']


In [8]:
# A flaky task: 'transform' fails the first time (transient), then succeeds on retry
attempts = {"transform": 0}                         # remember how many times each task ran
def run_task(name):
    if name == "transform":
        attempts[name] += 1                         # count this attempt
        if attempts[name] == 1:                     # fail only on the first attempt
            raise RuntimeError("transient timeout")
    return f"{name} ok"                             # otherwise the task succeeds

def run_with_retry(name, retries=2, base=0.02):     # retry wrapper with exponential backoff
    for attempt in range(retries + 1):              # initial try + `retries` more
        try:
            return run_task(name)                   # try to run the task
        except Exception as e:                      # a failure happened
            if attempt == retries:                  # out of retries -> give up (would alert)
                raise
            wait = base * (2 ** attempt)            # back off: wait longer each time
            print(f"  {name} failed ({e}); retry {attempt+1} after {wait:.2f}s")
            time.sleep(wait)                         # pause before retrying

# Execute the DAG in the computed order, retrying transient failures
print("Running the DAG:")
for t in run_order:                                 # run tasks respecting dependencies
    print(" ", run_with_retry(t))

Running the DAG:
  extract ok
  validate ok
  dedupe ok
  transform failed (transient timeout); retry 1 after 0.02s
  transform ok
  load ok
  publish ok


Wrap-up — reliability engineering
You built the plumbing the slides describe: a real ETL flow, idempotent loads, incremental extraction, a validation gate, date partitioning/backfills, and a tiny orchestrator (DAG + retries).

Takeaways

Transform is where most effort and bugs live — clean, cast, default, dedupe (Part 1).
Design every step to be idempotent so retries are safe (Part 2).
Pull only what changed with a high-water mark (Part 3).
Validate at the boundary; quarantine bad rows, don't let them through (Part 4).
Partition by date so re-runs and backfills are surgical (Part 5).
Orchestrators run a DAG in dependency order and retry failures (Part 6).
Real tools to graduate to: Apache Airflow / Prefect / Dagster (orchestration), dbt (SQL transforms + tests), Great Expectations (data quality).

Try yourself: add a quality_gate task between transform and load in Part 6's DAG and re-run topo_sort to see the order change.